# Imersão Alura Santander — Nivelamento IA 2026
## Semana 04 — Python S03 Continuado + OpenRouter Fallback

Este notebook acompanha a live da Semana 04.  
Todos os exemplos Python usam um cenário de **geração de copy em lote** — mais próximo do dia a dia de quem trabalha com marketing, agências e clientes.

**Estrutura:**
1. Setup — Provider Switcher (mesmo da S03)
2. Python S03 Cont. — Listas vs Tuplas
3. Python S03 Cont. — List Comprehension
4. Python S03 Cont. — Dict Comprehension
5. Python S03 Cont. — zip()
6. OpenRouter — Fallback Automático
7. Demo completa — pipeline de copy em lote

> **Sobre Cursor + Claude e Google AI Studio:** essas ferramentas são demonstradas ao vivo durante a aula.  
> Neste notebook você encontra as instruções de uso e os trechos de código usados nas demos.

---
## ⚙️ Setup — Provider Switcher

Mesmo setup da S03. Troque `PROVIDER` para alternar entre Groq e OpenRouter sem alterar mais nada.

**Execução local:** crie um arquivo `.env` nesta pasta (use `.env.example` como modelo) com `GROQ_API_KEY` e/ou `OPENROUTER_API_KEY`.

In [1]:
!pip install openai python-dotenv --quiet

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# Carrega GROQ_API_KEY / OPENROUTER_API_KEY do .env (pasta do notebook ou cwd)
load_dotenv(Path.cwd() / '.env')


def api_key(nome: str) -> str:
    valor = os.getenv(nome)
    if not valor:
        raise ValueError(
            f'{nome} não encontrada. '
            'Crie um arquivo .env nesta pasta (veja .env.example).'
        )
    return valor


# ─── ESCOLHA SEU PROVIDER ───────────────────────────────────────────────
PROVIDER = 'groq'   # troque para 'openrouter' e o restante funciona igual
# ────────────────────────────────────────────────────────────────────────

if PROVIDER == 'groq':
    client = OpenAI(
        base_url='https://api.groq.com/openai/v1',
        api_key=api_key('GROQ_API_KEY'),
    )
    MODEL = 'llama-3.3-70b-versatile'

elif PROVIDER == 'openrouter':
    client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=api_key('OPENROUTER_API_KEY'),
    )
    MODEL = 'meta-llama/llama-3.3-70b-instruct'

print(f'✅ Provider: {PROVIDER}')
print(f'   Modelo  : {MODEL}')

✅ Provider: groq
   Modelo  : llama-3.3-70b-versatile


---
## 🐍 Seção 3 — Listas vs Tuplas

**Conceito:** lista é mutável (cresce, muda); tupla é imutável (fixo em runtime).

| | Lista | Tupla |
|--|-------|-------|
| **Síntaxe** | `[a, b, c]` | `(a, b, c)` |
| **Mutável?** | Sim | Não |
| **Uso no cenário** | Copies geradas (cresce a cada produto) | Tons disponíveis (não mudam durante o script) |

> **Regra prática:** use tupla para comunicar "isso não deve mudar durante a execução".  
> Se alguém (ou você mesmo) tentar modificar, Python gera `TypeError` — e isso é bom.

In [2]:
# ─── CONFIGURAÇÃO IMUTÁVEL — tupla ──────────────────────────────────────
TONS_DISPONIVEIS = ('formal', 'descontraido', 'urgente', 'amigavel')
MODELOS_FALLBACK = ('groq', 'openrouter')   # providers ativos

# Tente modificar — vai gerar TypeError:
# TONS_DISPONIVEIS[0] = 'outro'   # ← descomente para ver o erro

print('Tons disponíveis:', TONS_DISPONIVEIS)
print('Tipo:', type(TONS_DISPONIVEIS))

# ─── LISTA DE RESULTADOS — mutável ──────────────────────────────────────
copies_geradas = []   # começa vazia, vai crescendo

def gerar_copy(produto: str, tom: str = 'descontraido') -> str:
    """Gera copy de marketing para um produto."""
    prompt = (
        f"Escreva um texto curto de marketing para o seguinte produto bancário.\n"
        f"Tom: {tom}. Máximo 2 frases. Termine com uma chamada para ação.\n"
        f"Produto: {produto}"
    )
    r = client.chat.completions.create(
        model=MODEL,
        temperature=0.7,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return r.choices[0].message.content.strip()

# Processar lista e acrescentar à lista de resultados
lista_produtos = [
    'Cartão de crédito sem anuidade com cashback',
    'Conta corrente para autônomos e MEIs',
    'CDB com liquidez diária e rendimento acima do CDI',
]

for produto in lista_produtos:
    copy = gerar_copy(produto, tom='descontraido')
    copies_geradas.append(copy)   # lista cresce a cada iteração

print(f'\nGeradas: {len(copies_geradas)} copies')
print(f'Tipo: {type(copies_geradas)}')

Tons disponíveis: ('formal', 'descontraido', 'urgente', 'amigavel')
Tipo: <class 'tuple'>

Geradas: 3 copies
Tipo: <class 'list'>


In [5]:
for copy in copies_geradas:
    print(f'  → {copy}')

  → Ganhe dinheiro de volta em suas compras com nosso cartão de crédito sem anuidade e cashback! Cadastre-se agora e comece a aproveitar os benefícios de ter dinheiro de volta no seu bolso!
  → Se você é autônomo ou MEI, sabe como pode ser complicado gerenciar suas finanças enquanto cuida do seu negócio! Abra uma conta corrente específica para autônomos e MEIs conosco e simplifique sua vida financeira - clique aqui para saber mais e abrir sua conta agora!
  → Investir nunca foi tão fácil e flexível! Abra agora mesmo um CDB com liquidez diária e rendimento acima do CDI conosco e comece a aproveitar os benefícios de uma rentabilidade superior sem comprometer sua liquidez.


## Exemplo Vibe Coding

In [10]:
# cria uma função que recebe o nome de um produto bancário e retorna a categoria: 'cartão', 'conta', 'seguro' ou 'investimento

In [ ]:
# crie uma funcao que calcula a média das notas de um aluno

---
## 🐍 Seção 4 — List Comprehension

**Conceito:** cria uma lista aplicando uma expressão a cada item de outra, em uma linha.

```python
# Estrutura geral
[expressão  for item in iterável  if condição_opcional]
```

> **No cenário de copy:** os produtos da lista são processados todos de uma vez.  
> Adicionar ou remover produtos da lista atualiza o processamento automaticamente — sem mudar a lógica.

In [4]:
# ─── SEM list comprehension — loop explícito ────────────────────────────
copies_loop = []
for p in lista_produtos:
    copies_loop.append(gerar_copy(p, tom='formal'))

# ─── COM list comprehension — a mesma coisa em uma linha ────────────────
copies = [gerar_copy(p, tom='formal') for p in lista_produtos]

print('Resultado com list comprehension:')
for copy in copies:
    print(f'  → {copy[:80]}...')

# ─── Com filtro: só produtos que mencionam 'conta' ───────────────────────
copies_conta = [
    gerar_copy(p)
    for p in lista_produtos
    if 'conta' in p.lower()
]
print(f'\nCopies de produtos do tipo "conta": {len(copies_conta)}')

Resultado com list comprehension:
  → Oferecemos um cartão de crédito sem anuidade que não apenas elimina as taxas anu...
  → Nossa conta corrente para autônomos e MEIs (Microempreendedores Individuais) foi...
  → O nosso CDB com liquidez diária oferece a combinação perfeita de flexibilidade e...

Copies de produtos do tipo "conta": 1


---
## 🐍 Seção 5 — Dict Comprehension

**Conceito:** cria um dicionário a partir de pares chave-valor com uma expressão.

```python
# Estrutura geral
{chave: valor  for item in iterável  if condição_opcional}
```

> **No cenário de copy:** cada tipo de produto recebe um system prompt diferente.  
> Dict comprehension monta essa tabela de roteamento dinamicamente.

In [5]:
# Base de instruções por tipo de produto — só edite aqui
instrucoes_base = {
    'cartao':       'Destaque benefícios e cashback. Tom animado e confiante.',
    'conta':        'Foque na praticidade e zero burocracia. Tom direto.',
    'investimento': 'Enfatize segurança e rentabilidade. Tom técnico e sólido.',
}

# Dict comprehension: montar prompts completos a partir das instruções
PREFIXO = 'Você escreve copy para um banco digital moderno. '
prompts_por_tipo = {
    tipo: PREFIXO + instrucao
    for tipo, instrucao in instrucoes_base.items()
}

print('System prompts por tipo:')
for tipo, prompt in prompts_por_tipo.items():
    print(f'  [{tipo}] {prompt[:70]}...')

# Função de roteamento: usa o prompt correto para cada tipo
def copy_roteada(produto: str, tipo: str) -> str:
    """Gera copy usando o system prompt específico do tipo de produto."""
    system = prompts_por_tipo.get(
        tipo,
        PREFIXO + 'Seja claro e objetivo. Máximo 2 frases.'
    )
    r = client.chat.completions.create(
        model=MODEL,
        temperature=0.7,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': f'Produto: {produto}\nEscreva em 2 frases com chamada para ação.'}
        ]
    )
    return r.choices[0].message.content.strip()

# Teste de roteamento
print('\n--- Teste de roteamento ---')
print(copy_roteada('CDB com liquidez diária', 'investimento'))

System prompts por tipo:
  [cartao] Você escreve copy para um banco digital moderno. Destaque benefícios e...
  [conta] Você escreve copy para um banco digital moderno. Foque na praticidade ...
  [investimento] Você escreve copy para um banco digital moderno. Enfatize segurança e ...

--- Teste de roteamento ---
Investir no nosso CDB com liquidez diária é uma escolha segura e rentável, pois oferece a combinação perfeita de proteção ao seu patrimônio e retorno financeiro atraente. Clique aqui para abrir sua conta e começar a investir agora, aproveitando as vantagens de nossa plataforma digital moderna e segura!


---
## 🐍 Seção 6 — zip()

**Conceito:** `zip()` emparelha dois (ou mais) iteráveis em pares sincronizados.

> **No cenário de copy:** `zip()` aparece toda vez que você precisa mostrar  
> a entrada (produto original) ao lado da saída (copy gerada), ou exportar para CSV.

In [6]:
# Produtos + tipos para roteamento
produtos_com_tipo = [
    ('Cartão de crédito sem anuidade',          'cartao'),
    ('Conta corrente para autônomos',            'conta'),
    ('CDB com liquidez diária',                  'investimento'),
    ('Seguro de vida com cobertura ampliada',    'outros'),
]

produtos = [p for p, _ in produtos_com_tipo]
tipos    = [t for _, t in produtos_com_tipo]

# Gerar copies roteadas com list comprehension
copies_finais = [copy_roteada(p, t) for p, t in zip(produtos, tipos)]

# ─── Exibir: zip() emparelha produto com copy ───────────────────────────
print('=== RESULTADO FINAL ===')
for produto, copy in zip(produtos, copies_finais):
    print(f'\nProduto : {produto}')
    print(f'Copy    : {copy[:120]}')

# ─── Exportar como CSV ──────────────────────────────────────────────────
import csv
with open('copies_geradas.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Produto', 'Tipo', 'Copy'])
    writer.writerows(zip(produtos, tipos, copies_finais))

print('\n✅ Exportado: copies_geradas.csv')
print('   Abra no Excel ou Google Sheets para visualizar.')

=== RESULTADO FINAL ===

Produto : Cartão de crédito sem anuidade
Copy    : Com o nosso cartão de crédito sem anuidade, você pode aproveitar todos os benefícios de um cartão de crédito premium sem

Produto : Conta corrente para autônomos
Copy    : Abra sua conta corrente para autônomos agora mesmo e esqueça a burocracia: com nossa plataforma digital, você pode geren

Produto : CDB com liquidez diária
Copy    : Investir com segurança e rentabilidade nunca foi tão fácil: nosso CDB com liquidez diária oferece a combinação perfeita 

Produto : Seguro de vida com cobertura ampliada
Copy    : Protija sua família e seu futuro com o nosso Seguro de Vida com Cobertura Ampliada, que oferece benefícios exclusivos e 

✅ Exportado: copies_geradas.csv
   Abra no Excel ou Google Sheets para visualizar.


---
## 🛠️ Demo: Cursor + Claude

Esta seção é demonstrada ao vivo durante a aula.  
Abaixo estão as instruções e os trechos de código usados.

### Demo 1 — Claude explicando código desconhecido

**Passos:**
1. Abrir este notebook no Cursor
2. Selecionar o trecho com `list comprehension + filtro` da seção 4
3. `Ctrl+L` → abrir chat → digitar:
   ```
   Explica esse código linha por linha em português simples
   ```
4. Follow-up: `"E se eu quiser filtrar por dois critérios ao mesmo tempo?"`

---

### Demo 2 — Claude adaptando código para novo caso

**Passos:**
1. Selecionar a função `gerar_copy` e a list comprehension
2. `Ctrl+K` → digitar:
   ```
   Adapta esse código para ler os produtos de um arquivo CSV
   chamado produtos.csv em vez de uma lista fixa
   ```
3. Rodar e verificar

O código gerado pelo Claude ficará parecido com a célula abaixo:

In [ ]:
# ─── Versão adaptada pelo Claude para ler de CSV ─────────────────────────
# (Este código seria gerado pela demo do Cursor — está aqui para referência)

import csv

def processar_csv(caminho_csv: str) -> list[dict]:
    """
    Lê produtos de um CSV e gera copy para cada um.
    O CSV deve ter colunas: produto, tipo
    """
    resultados = []
    with open(caminho_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            copy = copy_roteada(row['produto'], row.get('tipo', 'outros'))
            resultados.append({
                'produto': row['produto'],
                'tipo':    row.get('tipo', 'outros'),
                'copy':    copy
            })
    return resultados

# Para testar: crie um arquivo produtos.csv com colunas produto,tipo
# e descomente a linha abaixo:
# resultados = processar_csv('produtos.csv')

print('Função processar_csv() definida.')
print('Crie produtos.csv e chame processar_csv("produtos.csv") para testar.')

---
## 🔁 OpenRouter — Fallback Automático

**Contexto:** na aula da Semana 03 não deu tempo de mostrar esse exemplo.  
O conceito foi mencionado no slide do OpenRouter, mas sem código.

### O problema

Em produção, modelos ficam fora do ar: rate limit, sobrecarga, manutenção.  
Sem fallback, o agente para. Com o header `X-OR-Fallback-Models`, o OpenRouter redireciona automaticamente.

### Para quem está aprendendo

Se o Groq der rate limit nos seus testes, use o **fallback manual** — troque `PROVIDER = 'openrouter'` na célula de setup.  
O fallback automático via header é para quando você tiver um sistema em produção que não pode parar.

In [ ]:
# ─── Fallback automático via header ─────────────────────────────────────
# Requer PROVIDER = 'openrouter' na célula de setup

from openai import OpenAI

client_fallback = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=api_key('OPENROUTER_API_KEY'),
)

def chamar_com_fallback(mensagem: str) -> str:
    """
    Chama o modelo com lista de fallback automático.
    Ordem de prioridade:
      1. meta-llama/llama-3.3-70b-instruct  (gratuito — modelo principal)
      2. google/gemma-3-27b-it:free          (gratuito — fallback 1)
      3. mistralai/mistral-7b-instruct:free  (gratuito — fallback 2)
      4. openai/gpt-4o-mini                  (pago — último recurso)
    """
    r = client_fallback.chat.completions.create(
        model='meta-llama/llama-3.3-70b-instruct',
        messages=[{'role': 'user', 'content': mensagem}],
        extra_headers={
            'X-OR-Fallback-Models': (
                'google/gemma-3-27b-it:free,'
                'mistralai/mistral-7b-instruct:free,'
                'openai/gpt-4o-mini'
            )
        }
    )
    return r.choices[0].message.content

# Teste (requer OPENROUTER_API_KEY no .env e célula de setup executada)
# resposta = chamar_com_fallback('Olá! Qual modelo você é?')
# print(resposta)

print('Função chamar_com_fallback() definida.')
print('Configure OPENROUTER_API_KEY no .env e descomente o teste.')

---
## 🏆 Pipeline Completo — Copy em Lote

Aqui reunimos tudo que construímos na live:  
listas/tuplas, list comprehension, dict comprehension e zip() em um pipeline funcional.

| Conceito | Papel no pipeline |
|----------|------------------|
| Tupla | Configuração imutável de tons e providers |
| Lista | Produtos de entrada e copies geradas |
| List Comprehension | Gerar todas as copies em uma linha |
| Dict Comprehension | Montar system prompts por tipo de produto |
| zip() | Emparelhar produto + tipo + copy para exportar |

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PIPELINE COMPLETO — COPY EM LOTE
# ═══════════════════════════════════════════════════════════════

# 1. Configuração imutável (tupla)
TONS_OK      = ('formal', 'descontraido', 'urgente')
TOM_PADRAO   = 'descontraido'

# 2. Base de instruções (dict comprehension vai usar isso)
instrucoes = {
    'cartao':       'Destaque cashback e benefícios. Tom animado.',
    'conta':        'Foque em praticidade. Tom direto e moderno.',
    'investimento': 'Enfatize segurança e rentabilidade. Tom técnico.',
}
PREFIXO = 'Você cria copy para banco digital. 2 frases + chamada para ação. '
system_prompts = { tipo: PREFIXO + inst for tipo, inst in instrucoes.items() }  # dict comprehension

# 3. Dados de entrada (lista)
catalogo = [
    {'produto': 'Cartão sem anuidade',              'tipo': 'cartao'},
    {'produto': 'Conta para autônomos e MEIs',       'tipo': 'conta'},
    {'produto': 'CDB com liquidez diária',           'tipo': 'investimento'},
    {'produto': 'Seguro de vida para famílias',      'tipo': 'outros'},
]

# 4. Função de geração com roteamento
def gerar_copy_roteada(item: dict) -> str:
    system = system_prompts.get(item['tipo'], PREFIXO + 'Seja claro e objetivo.')
    r = client.chat.completions.create(
        model=MODEL, temperature=0.7,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': f"Produto: {item['produto']}"}
        ]
    )
    return r.choices[0].message.content.strip()

# 5. Processar lote (list comprehension)
copies = [gerar_copy_roteada(item) for item in catalogo]

# 6. Exibir (zip)
print(f'=== PIPELINE [{PROVIDER.upper()}] — {len(catalogo)} produtos ===')
for item, copy in zip(catalogo, copies):
    print(f"\n[{item['tipo'].upper()}] {item['produto']}")
    print(f"  → {copy[:120]}")

# 7. Exportar CSV
with open('copies_finais.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Produto', 'Tipo', 'Copy'])
    writer.writerows(
        (item['produto'], item['tipo'], copy)
        for item, copy in zip(catalogo, copies)
    )
print('\n✅ Exportado: copies_finais.csv')

---
## 📝 Exercícios da Semana 04

**Iniciante**  
Abra o Google AI Studio (aistudio.google.com), configure uma System Instruction para o seu contexto profissional e teste com 5 textos diferentes. Observe como o tom se mantém consistente entre as mensagens.

**Intermediário**  
Adapte a função `gerar_copy` para ler produtos de um arquivo CSV real (`produtos.csv` com colunas `produto` e `tipo`). Use a função `processar_csv()` definida na seção de demo do Cursor como referência.

**Avançado**  
Implemente o fallback do OpenRouter diretamente no Provider Switcher:  
- Quando `PROVIDER = 'groq'`, a função deve tentar Groq primeiro  
- Se receber `RateLimitError`, deve redirecionar automaticamente para OpenRouter  
- Use `try/except` (prévia da S05) + a lista de fallback do header `X-OR-Fallback-Models`

---

**Links úteis:**
- [Groq Console](https://console.groq.com)
- [OpenRouter — modelos gratuitos](https://openrouter.ai/models?q=free)
- [Google AI Studio](https://aistudio.google.com)
- [OpenRouter Docs — Fallback](https://openrouter.ai/docs/features/fallbacks)